In [1]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

from networks import GAT, PPGAT
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch_geometric.data import DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score


env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [2]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

torch.use_deterministic_algorithms(mode=True)

In [3]:
target = "Q16790"
dataset = torch.load(f'datasets/Gdatasets/{target}_dataset.pt', weights_only=False)
rg_dataset = torch.load(f'datasets/RGdatasets/{target}_RG_dataset.pt', weights_only=False)
ppgat_dataset = torch.load(f'datasets/PPGATdatasets/{target}_PPGAT_dataset.pt', weights_only=False)

In [4]:
#split data 
train_val, test_data = train_test_split(dataset, test_size=0.2)
train_data, val_data = train_test_split(train_val, test_size=0.125)

train_loader = DataLoader(train_data,batch_size=32)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)




/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

in_channels = dataset[0].x.size(-1)
edge_attr_dim = dataset[0].edge_attr.size(-1)

model = GAT(in_channels, edge_attr_dim, hidden_channels=64, out_channels=1).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()




In [6]:
for epoch in range(1,100):
    model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y.float().unsqueeze(1))
        loss.backward()
        optimizer.step()
    print(f"Epoch{epoch}, Train Loss:{loss:.4f}")
    
       

Epoch1, Train Loss:0.1079
Epoch2, Train Loss:0.0835
Epoch3, Train Loss:0.1033
Epoch4, Train Loss:0.1277
Epoch5, Train Loss:0.1282
Epoch6, Train Loss:0.1284
Epoch7, Train Loss:0.1174
Epoch8, Train Loss:0.0994
Epoch9, Train Loss:0.1128
Epoch10, Train Loss:0.1345
Epoch11, Train Loss:0.1278
Epoch12, Train Loss:0.0977
Epoch13, Train Loss:0.0718
Epoch14, Train Loss:0.0705
Epoch15, Train Loss:0.0377
Epoch16, Train Loss:0.0231
Epoch17, Train Loss:0.0280
Epoch18, Train Loss:0.0374
Epoch19, Train Loss:0.0074
Epoch20, Train Loss:0.0139
Epoch21, Train Loss:0.0106
Epoch22, Train Loss:0.0371
Epoch23, Train Loss:0.0104
Epoch24, Train Loss:0.0143
Epoch25, Train Loss:0.0098
Epoch26, Train Loss:0.0331
Epoch27, Train Loss:0.0086
Epoch28, Train Loss:0.0098
Epoch29, Train Loss:0.0117
Epoch30, Train Loss:0.0150
Epoch31, Train Loss:0.0361
Epoch32, Train Loss:0.0130
Epoch33, Train Loss:0.0308
Epoch34, Train Loss:0.0345
Epoch35, Train Loss:0.0578
Epoch36, Train Loss:0.1029
Epoch37, Train Loss:0.0393
Epoch38, T

In [7]:
def evaluate_model(model, dataloader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for data in dataloader:
            data = data.to(device)
            out = model(data)
            probs = torch.sigmoid(out).view(-1).cpu().numpy()
            labels = data.y.view(-1).cpu().numpy()

            # Ensure probs and labels are arrays, even for batch size 1
            probs = np.atleast_1d(probs)
            labels = np.atleast_1d(labels)

            all_probs.extend(probs)
            all_labels.extend(labels)

    y_pred = (np.array(all_probs) > 0.5).astype(int)
    acc = accuracy_score(all_labels, y_pred)
    auroc = roc_auc_score(all_labels, all_probs)
    return acc, auroc

In [9]:
final_acc, final_auroc = evaluate_model(model, test_loader, device)
print("AUROC: ", final_auroc, "Accuracy: ", final_acc)

AUROC:  0.9882076752123881 Accuracy:  0.966931216931217


In [9]:
final_acc, final_auroc = evaluate_model(model, test_loader, device)
print("AUROC: ", final_auroc, "Accuracy: ", final_acc)

AUROC:  0.9882076752123881 Accuracy:  0.966931216931217
